# 09 — Aplicação Estratégica e Inteligência Educacional

## Objetivo

Este notebook transforma as previsões e os artefatos produzidos no Notebook 08 em análises territoriais aplicáveis ao contexto do Tech Challenge — Fase 3.

- ranking municipal de risco educacional;
- comparação entre alfabetização prevista e meta municipal;
- identificação de municípios com maior risco de não atingir a meta;
- agrupamento de municípios com padrões semelhantes;
- geração de artefatos e visualizações executivas.

## Limite de interpretação

As análises utilizam os **834 municípios do conjunto de teste**, que não participaram do treinamento nem da seleção do modelo. Portanto, os rankings representam uma demonstração estratégica fora da amostra, e não um ranking nacional completo.

O notebook não retreina o modelo, não altera o limiar e não modifica a seleção do HistGradientBoosting.


## 1. Configuração e caminhos

Os artefatos de modelagem serão lidos exclusivamente da estrutura da Fase 3. A base Gold da Fase 2 será utilizada somente para recuperar a dimensão municipal com nome e sigla da UF.

Todas as novas tabelas e imagens serão gravadas em diretórios próprios da aplicação estratégica na Fase 3.


In [0]:
# Objetivo:
#
# Configurar o ambiente e os caminhos oficiais.
#
# Justificativa:
#
# A separação entre entrada e saída preserva a Gold
# da Fase 2 e concentra os novos artefatos na Fase 3.
#
# Ação:
#
# Importa bibliotecas, define sementes, parâmetros
# e cria os diretórios de relatórios e imagens.

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


warnings.filterwarnings(
    "ignore",
    category=FutureWarning
)

RANDOM_STATE = 42
MIN_ALUNOS_MUNICIPIO = 100
MARGEM_ATENCAO_META_PP = 5.0

FASE2_ROOT = (
    "/Volumes/workspace/default/vol_trio_drive/"
    "projetos/fiap/tech_challenge_fase2"
)

FASE3_ROOT = (
    "/Volumes/workspace/default/vol_trio_drive/"
    "projetos/fiap/tech_challenge_fase3"
)

MODELAGEM_PATH = f"{FASE3_ROOT}/modelagem"
RELATORIOS_MODELAGEM_PATH = (
    f"{FASE3_ROOT}/reports/modelagem"
)
APLICACAO_PATH = (
    f"{FASE3_ROOT}/reports/aplicacao_estrategica"
)
IMAGENS_PATH = (
    f"{FASE3_ROOT}/images/aplicacao_estrategica"
)

BASE_FINAL_PATH = (
    f"{FASE2_ROOT}/gold/"
    "alunos_base_enriquecida_finalizada/"
    "base_analitica_final.csv"
)

PREVISOES_PATH = (
    f"{RELATORIOS_MODELAGEM_PATH}/"
    "previsoes_teste_rastreaveis.parquet"
)

TESTE_PATH = f"{MODELAGEM_PATH}/teste.parquet"
METADATA_PATH = (
    f"{FASE3_ROOT}/artifacts/model_metadata.json"
)

dbutils.fs.mkdirs(APLICACAO_PATH)
dbutils.fs.mkdirs(IMAGENS_PATH)

sns.set_theme(style="whitegrid", context="notebook")

print("Entrada de previsões:", PREVISOES_PATH)
print("Conjunto de teste:", TESTE_PATH)
print("Saídas tabulares:", APLICACAO_PATH)
print("Imagens:", IMAGENS_PATH)


## 2. Carregamento e validação dos artefatos

O conjunto de teste contém as 16 features e o target. O artefato de previsões contém os identificadores auxiliares, a probabilidade de risco, a classe prevista e o tipo de resultado.

As estruturas devem possuir a mesma quantidade e ordem de registros. A validação interrompe o notebook se houver qualquer divergência.


In [0]:
# Objetivo:
#
# Carregar os artefatos produzidos pelos Notebooks
# 07 e 08 e validar o contrato de correspondência.
#
# Justificativa:
#
# O ranking municipal depende do alinhamento exato
# entre features, target, previsões e identificadores.
#
# Ação:
#
# Carrega os Parquets e metadados e executa testes
# estruturais antes das análises estratégicas.

teste = pd.read_parquet(TESTE_PATH).reset_index(drop=True)
previsoes = pd.read_parquet(
    PREVISOES_PATH
).reset_index(drop=True)

with open(
    METADATA_PATH,
    "r",
    encoding="utf-8"
) as arquivo:
    metadados_modelo = json.load(arquivo)


colunas_previsoes_obrigatorias = {
    "id_aluno",
    "id_escola",
    "co_municipio",
    "y_real",
    "probabilidade_nao_alfabetizado",
    "y_pred",
    "limiar_aplicado",
    "tipo_resultado"
}

colunas_teste_obrigatorias = {
    "in_alfabetizado",
    "co_uf",
    "tp_dependencia",
    "indicador_meta_final_2025",
    "indicador_pc_aluno_alfabetizado_2024"
}

faltantes_previsoes = sorted(
    colunas_previsoes_obrigatorias
    - set(previsoes.columns)
)

faltantes_teste = sorted(
    colunas_teste_obrigatorias
    - set(teste.columns)
)

if faltantes_previsoes or faltantes_teste:
    raise ValueError(
        "Contrato de entrada inválido. "
        f"Previsões ausentes: {faltantes_previsoes}. "
        f"Teste ausentes: {faltantes_teste}."
    )

if len(teste) != len(previsoes):
    raise ValueError(
        "Teste e previsões possuem quantidades "
        "diferentes de registros."
    )

target_alinhado = np.array_equal(
    teste["in_alfabetizado"].to_numpy(),
    previsoes["y_real"].to_numpy()
)

if not target_alinhado:
    raise ValueError(
        "A ordem dos registros não corresponde entre "
        "o teste e as previsões rastreáveis."
    )

validacao_entrada = pd.DataFrame([{
    "registros_teste": int(len(teste)),
    "registros_previsoes": int(len(previsoes)),
    "municipios": int(
        previsoes["co_municipio"].nunique()
    ),
    "target_alinhado": bool(target_alinhado),
    "modelo": str(metadados_modelo["modelo"]),
    "limiar": float(metadados_modelo["limiar"])
}])

display(validacao_entrada)


### 2.1 Construção da base estratégica em nível de aluno

As features necessárias serão anexadas às previsões pela posição já validada. Os identificadores continuam sendo usados somente para rastreabilidade e agregação, nunca como entrada do modelo.


In [0]:
# Objetivo:
#
# Reunir previsões, target e contexto municipal.
#
# Justificativa:
#
# A análise estratégica precisa combinar risco,
# meta e características territoriais sem utilizar
# identificadores como features preditivas.
#
# Ação:
#
# Cria uma base derivada e padroniza códigos.

base_estrategica_aluno = previsoes.copy()

colunas_contextuais = [
    coluna
    for coluna in teste.columns
    if coluna != "in_alfabetizado"
]

for coluna in colunas_contextuais:
    base_estrategica_aluno[coluna] = (
        teste[coluna].to_numpy()
    )


def normalizar_codigo(serie):
    return (
        serie
        .astype("string")
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
    )


base_estrategica_aluno["co_municipio"] = (
    normalizar_codigo(
        base_estrategica_aluno["co_municipio"]
    )
)

base_estrategica_aluno["co_uf"] = (
    normalizar_codigo(
        base_estrategica_aluno["co_uf"]
    )
)

base_estrategica_aluno["alerta_risco"] = (
    base_estrategica_aluno["y_pred"].eq(0)
    .astype("int8")
)

base_estrategica_aluno["nao_alfabetizado_real"] = (
    base_estrategica_aluno["y_real"].eq(0)
    .astype("int8")
)

base_estrategica_aluno["falso_negativo"] = (
    base_estrategica_aluno["tipo_resultado"]
    .eq("FN_nao_alfabetizado")
    .astype("int8")
)

base_estrategica_aluno.shape


### 2.2 Dimensão municipal

Para tornar os rankings legíveis, o notebook recupera apenas o código, nome do município e sigla da UF na base final. Essa leitura não modifica a Gold e não incorpora novas informações ao modelo.


In [0]:
# Objetivo:
#
# Recuperar nomes dos municípios e siglas das UFs.
#
# Justificativa:
#
# Códigos são adequados para integração, mas nomes
# tornam os relatórios executivos compreensíveis.
#
# Ação:
#
# Lê somente três colunas da Gold, remove duplicidades
# e cria uma dimensão municipal para enriquecimento.

dimensao_municipio = pd.read_csv(
    BASE_FINAL_PATH,
    sep=";",
    encoding="utf-8-sig",
    usecols=[
        "co_municipio",
        "no_municipio",
        "sg_uf"
    ],
    low_memory=False
)

dimensao_municipio["co_municipio"] = (
    normalizar_codigo(
        dimensao_municipio["co_municipio"]
    )
)

dimensao_municipio = (
    dimensao_municipio
    .dropna(subset=["co_municipio"])
    .drop_duplicates(subset=["co_municipio"])
    .reset_index(drop=True)
)

if dimensao_municipio["co_municipio"].duplicated().any():
    raise ValueError(
        "A dimensão municipal possui códigos duplicados."
    )

display(dimensao_municipio.head())


## 3. Construção do painel municipal de risco

O painel transforma previsões individuais em indicadores agregados por município. Para preservar estabilidade, os rankings principais consideram somente municípios com pelo menos 100 alunos no conjunto de teste.

A probabilidade média de não alfabetização representa o risco predito médio. A taxa prevista de alfabetização é calculada como `1 - probabilidade média de risco`.


In [0]:
# Objetivo:
#
# Construir indicadores agregados por município.
#
# Justificativa:
#
# A aplicação estratégica exige uma unidade compatível
# com políticas territoriais e metas municipais.
#
# Ação:
#
# Agrega volume, risco, alertas, alfabetização real,
# previsão probabilística e falsos negativos.

def primeiro_valor_valido(serie):
    valores = serie.dropna()
    return valores.iloc[0] if len(valores) else np.nan


agregacoes_municipais = {
    "id_aluno": "size",
    "probabilidade_nao_alfabetizado": [
        "mean",
        "median",
        lambda s: s.quantile(0.90)
    ],
    "alerta_risco": "mean",
    "nao_alfabetizado_real": "mean",
    "falso_negativo": ["sum", "mean"],
    "co_uf": primeiro_valor_valido,
    "tp_dependencia": lambda s: (
        s.mode().iloc[0] if not s.mode().empty else np.nan
    )
}

features_contextuais_municipais = [
    "atlas_idhm",
    "atlas_idhm_e",
    "atlas_renda_pc",
    "atlas_indice_gini",
    "atlas_prop_pobreza_criancas",
    "atlas_taxa_criancas_dom_sem_fund",
    "censo_prop_mat_2ano_internet_aprendizagem",
    "censo_prop_mat_2ano_alimentacao",
    "censo_prop_mat_2ano_biblioteca_sala_leitura",
    "fundeb_receita_contribuicao",
    "fundeb_complementacao_uniao",
    "fundeb_receita_total",
    "indicador_meta_final_2025",
    "indicador_pc_aluno_alfabetizado_2024"
]

for coluna in features_contextuais_municipais:
    agregacoes_municipais[coluna] = "median"


painel_municipal = (
    base_estrategica_aluno
    .groupby("co_municipio", dropna=False)
    .agg(agregacoes_municipais)
)

painel_municipal.columns = [
    "_".join(
        str(parte)
        for parte in coluna
        if str(parte)
    ).rstrip("_")
    if isinstance(coluna, tuple)
    else str(coluna)
    for coluna in painel_municipal.columns
]

renomear_agregacoes = {
    "id_aluno_size": "quantidade_alunos",
    "probabilidade_nao_alfabetizado_mean": (
        "probabilidade_risco_media"
    ),
    "probabilidade_nao_alfabetizado_median": (
        "probabilidade_risco_mediana"
    ),
    "probabilidade_nao_alfabetizado_<lambda_0>": (
        "probabilidade_risco_p90"
    ),
    "alerta_risco_mean": "taxa_alertas",
    "nao_alfabetizado_real_mean": (
        "taxa_nao_alfabetizado_real"
    ),
    "falso_negativo_sum": "falsos_negativos",
    "falso_negativo_mean": "taxa_falsos_negativos",
    "co_uf_primeiro_valor_valido": "co_uf",
    "tp_dependencia_<lambda>": "dependencia_predominante"
}

painel_municipal = (
    painel_municipal
    .rename(columns=renomear_agregacoes)
    .reset_index()
)

coluna_dependencia = [
    coluna
    for coluna in painel_municipal.columns
    if coluna.startswith("tp_dependencia_")
]

if coluna_dependencia:
    painel_municipal = painel_municipal.rename(
        columns={
            coluna_dependencia[0]: (
                "dependencia_predominante"
            )
        }
    )

coluna_p90 = [
    coluna
    for coluna in painel_municipal.columns
    if coluna.startswith(
        "probabilidade_nao_alfabetizado_"
    )
]

if coluna_p90:
    painel_municipal = painel_municipal.rename(
        columns={
            coluna_p90[0]: "probabilidade_risco_p90"
        }
    )

colunas_agregadas_obrigatorias = {
    "quantidade_alunos",
    "probabilidade_risco_media",
    "probabilidade_risco_mediana",
    "probabilidade_risco_p90",
    "taxa_alertas",
    "taxa_nao_alfabetizado_real",
    "falsos_negativos",
    "taxa_falsos_negativos",
    "co_uf",
    "dependencia_predominante"
}

faltantes_agregacao = sorted(
    colunas_agregadas_obrigatorias
    - set(painel_municipal.columns)
)

if faltantes_agregacao:
    raise ValueError(
        "Falha no contrato da agregação municipal. "
        f"Colunas ausentes: {faltantes_agregacao}."
    )

painel_municipal["taxa_alfabetizacao_real_pct"] = (
    100
    * (1 - painel_municipal[
        "taxa_nao_alfabetizado_real"
    ])
)

painel_municipal[
    "taxa_alfabetizacao_predita_prob_pct"
] = (
    100
    * (1 - painel_municipal[
        "probabilidade_risco_media"
    ])
)

painel_municipal["taxa_alertas_pct"] = (
    100 * painel_municipal["taxa_alertas"]
)

painel_municipal = painel_municipal.merge(
    dimensao_municipio,
    on="co_municipio",
    how="left",
    validate="one_to_one"
)

painel_municipal["rotulo_municipio"] = (
    painel_municipal["no_municipio"]
    .fillna(painel_municipal["co_municipio"])
    .astype(str)
    + " - "
    + painel_municipal["sg_uf"]
    .fillna(painel_municipal["co_uf"])
    .astype(str)
)

painel_municipal_elegivel = (
    painel_municipal.loc[
        painel_municipal["quantidade_alunos"]
        >= MIN_ALUNOS_MUNICIPIO
    ]
    .copy()
)

validacao_painel = pd.DataFrame([{
    "municipios_teste": int(len(painel_municipal)),
    "municipios_elegiveis_ranking": int(
        len(painel_municipal_elegivel)
    ),
    "minimo_alunos_ranking": int(
        MIN_ALUNOS_MUNICIPIO
    ),
    "alunos_representados": int(
        painel_municipal["quantidade_alunos"].sum()
    )
}])

display(validacao_painel)


## 4. Ranking municipal de risco educacional

O ranking é ordenado pela probabilidade média prevista de não alfabetização. As faixas de risco são relativas aos municípios elegíveis do conjunto de teste:

- alta: 30% superiores;
- moderada: faixa intermediária;
- baixa: 40% inferiores.

Essas faixas priorizam investigação e não representam diagnóstico ou causalidade.


In [0]:
# Objetivo:
#
# Ordenar os municípios por risco previsto.
#
# Justificativa:
#
# A probabilidade média preserva informação que seria
# perdida ao utilizar apenas a classe no limiar.
#
# Ação:
#
# Calcula percentil relativo, cria faixas e apresenta
# os vinte municípios de maior risco previsto.

painel_municipal_elegivel[
    "percentil_risco_relativo"
] = (
    painel_municipal_elegivel[
        "probabilidade_risco_media"
    ]
    .rank(method="average", pct=True)
)

painel_municipal_elegivel["faixa_risco_relativo"] = (
    pd.cut(
        painel_municipal_elegivel[
            "percentil_risco_relativo"
        ],
        bins=[0.0, 0.40, 0.70, 1.0],
        labels=["Baixo", "Moderado", "Alto"],
        include_lowest=True
    )
)

ranking_municipal_risco = (
    painel_municipal_elegivel
    .sort_values(
        [
            "probabilidade_risco_media",
            "quantidade_alunos"
        ],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

ranking_municipal_risco.insert(
    0,
    "posicao_risco",
    np.arange(1, len(ranking_municipal_risco) + 1)
)

colunas_ranking = [
    "posicao_risco",
    "co_municipio",
    "no_municipio",
    "sg_uf",
    "quantidade_alunos",
    "probabilidade_risco_media",
    "probabilidade_risco_p90",
    "taxa_alertas_pct",
    "taxa_alfabetizacao_real_pct",
    "falsos_negativos",
    "faixa_risco_relativo"
]

display(
    ranking_municipal_risco[
        colunas_ranking
    ].head(20)
)


In [0]:
# Objetivo:
#
# Visualizar os municípios com maior risco previsto.
#
# Justificativa:
#
# O gráfico facilita a leitura executiva do ranking.
#
# Ação:
#
# Exibe os vinte primeiros e salva a imagem na Fase 3.

top_risco = (
    ranking_municipal_risco.head(20)
    .sort_values("probabilidade_risco_media")
)

fig, ax = plt.subplots(figsize=(11, 8))

ax.barh(
    top_risco["rotulo_municipio"],
    100 * top_risco["probabilidade_risco_media"],
    color="#C44E52"
)

ax.set_title(
    "Municípios do teste com maior risco médio previsto"
)
ax.set_xlabel("Probabilidade média de não alfabetização (%)")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.25)

plt.tight_layout()
plt.savefig(
    f"{IMAGENS_PATH}/top20_risco_municipal.png",
    dpi=160,
    bbox_inches="tight"
)
plt.show()


## 5. Gap previsto para a meta municipal

As colunas de meta e indicador anterior podem estar registradas em escala de 0 a 1 ou de 0 a 100. A função abaixo detecta a escala e converte os valores para percentual.

O gap é calculado como:

```text
alfabetização prevista (%) - meta municipal (%)
```

Valores negativos indicam risco de não atingir a meta. A classificação estratégica utiliza uma margem de cinco pontos percentuais apenas como regra comunicacional do projeto.


In [0]:
# Objetivo:
#
# Padronizar indicadores percentuais e calcular o
# gap entre alfabetização prevista e meta municipal.
#
# Justificativa:
#
# Uma diferença de escala produziria conclusões
# incorretas sobre o atingimento das metas.
#
# Ação:
#
# Detecta a escala, converte para percentual e
# classifica a situação estratégica do município.

def converter_para_percentual(serie, nome):
    serie_numerica = pd.to_numeric(
        serie,
        errors="coerce"
    )

    valores_validos = serie_numerica.dropna()

    if valores_validos.empty:
        raise ValueError(
            f"A variável {nome} não possui valores válidos."
        )

    maximo = float(valores_validos.max())
    minimo = float(valores_validos.min())

    if minimo < 0:
        raise ValueError(
            f"A variável {nome} contém valores negativos."
        )

    if maximo <= 1.5:
        return 100 * serie_numerica, "proporcao_0_1"

    if maximo <= 100.0:
        return serie_numerica, "percentual_0_100"

    raise ValueError(
        f"Escala inesperada em {nome}: máximo {maximo}."
    )


(
    painel_municipal_elegivel["meta_2025_pct"],
    escala_meta
) = converter_para_percentual(
    painel_municipal_elegivel[
        "indicador_meta_final_2025_median"
    ],
    "indicador_meta_final_2025"
)

(
    painel_municipal_elegivel[
        "alfabetizacao_2024_pct"
    ],
    escala_indicador_2024
) = converter_para_percentual(
    painel_municipal_elegivel[
        "indicador_pc_aluno_alfabetizado_2024_median"
    ],
    "indicador_pc_aluno_alfabetizado_2024"
)

painel_municipal_elegivel["gap_previsto_meta_pp"] = (
    painel_municipal_elegivel[
        "taxa_alfabetizacao_predita_prob_pct"
    ]
    - painel_municipal_elegivel["meta_2025_pct"]
)

condicoes_meta = [
    painel_municipal_elegivel[
        "gap_previsto_meta_pp"
    ].lt(-MARGEM_ATENCAO_META_PP),
    painel_municipal_elegivel[
        "gap_previsto_meta_pp"
    ].lt(0)
]

rotulos_meta = [
    "Risco elevado de não atingir",
    "Atenção — abaixo da meta"
]

painel_municipal_elegivel["situacao_meta"] = np.select(
    condicoes_meta,
    rotulos_meta,
    default="Meta prevista como atingida"
)

municipios_risco_meta = (
    painel_municipal_elegivel
    .sort_values("gap_previsto_meta_pp")
    .reset_index(drop=True)
)

validacao_escala = pd.DataFrame([{
    "escala_meta": escala_meta,
    "escala_indicador_2024": escala_indicador_2024,
    "margem_atencao_pp": float(
        MARGEM_ATENCAO_META_PP
    ),
    "municipios_abaixo_meta": int(
        municipios_risco_meta[
            "gap_previsto_meta_pp"
        ].lt(0).sum()
    ),
    "municipios_risco_elevado": int(
        municipios_risco_meta[
            "gap_previsto_meta_pp"
        ].lt(-MARGEM_ATENCAO_META_PP).sum()
    )
}])

display(validacao_escala)

display(
    municipios_risco_meta[[
        "co_municipio",
        "no_municipio",
        "sg_uf",
        "quantidade_alunos",
        "alfabetizacao_2024_pct",
        "meta_2025_pct",
        "taxa_alfabetizacao_predita_prob_pct",
        "gap_previsto_meta_pp",
        "probabilidade_risco_media",
        "situacao_meta"
    ]].head(20)
)


In [0]:
# Objetivo:
#
# Visualizar os maiores déficits previstos para a meta.
#
# Justificativa:
#
# O gap em pontos percentuais combina a previsão do
# modelo com a referência estratégica municipal.
#
# Ação:
#
# Exibe os vinte menores gaps e salva a figura.

top_gap_negativo = (
    municipios_risco_meta.head(20)
    .sort_values("gap_previsto_meta_pp", ascending=False)
)

cores_gap = np.where(
    top_gap_negativo["gap_previsto_meta_pp"] < 0,
    "#C44E52",
    "#55A868"
)

fig, ax = plt.subplots(figsize=(11, 8))

ax.barh(
    top_gap_negativo["rotulo_municipio"],
    top_gap_negativo["gap_previsto_meta_pp"],
    color=cores_gap
)

ax.axvline(0, color="black", linewidth=1)
ax.set_title(
    "Municípios do teste com menor gap previsto para a meta"
)
ax.set_xlabel("Gap previsto para a meta (pontos percentuais)")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.25)

plt.tight_layout()
plt.savefig(
    f"{IMAGENS_PATH}/top20_gap_meta.png",
    dpi=160,
    bbox_inches="tight"
)
plt.show()


## 6. Clusterização de municípios com padrões semelhantes

A clusterização é uma análise não supervisionada complementar. Ela não altera o classificador e não utiliza o target real como entrada.

As variáveis do agrupamento representam risco previsto, meta, histórico educacional, desenvolvimento, vulnerabilidade, infraestrutura e financiamento. Receitas do Fundeb recebem `log1p` para reduzir assimetria.

O número de clusters será escolhido entre 2 e 8 pelo maior silhouette score.


In [0]:
# Objetivo:
#
# Preparar as variáveis municipais para clusterização.
#
# Justificativa:
#
# Escalas diferentes e valores ausentes exigem
# tratamento antes do KMeans.
#
# Ação:
#
# Cria receitas em log1p e define o conjunto de
# variáveis sem utilizar o target real.

base_clusters = painel_municipal_elegivel.copy()

colunas_fundeb = [
    "fundeb_receita_contribuicao_median",
    "fundeb_complementacao_uniao_median",
    "fundeb_receita_total_median"
]

for coluna in colunas_fundeb:
    base_clusters[f"log1p_{coluna}"] = np.log1p(
        pd.to_numeric(
            base_clusters[coluna],
            errors="coerce"
        ).clip(lower=0)
    )

features_cluster = [
    "probabilidade_risco_media",
    "meta_2025_pct",
    "alfabetizacao_2024_pct",
    "atlas_idhm_median",
    "atlas_idhm_e_median",
    "atlas_renda_pc_median",
    "atlas_indice_gini_median",
    "atlas_prop_pobreza_criancas_median",
    "atlas_taxa_criancas_dom_sem_fund_median",
    "censo_prop_mat_2ano_internet_aprendizagem_median",
    "censo_prop_mat_2ano_alimentacao_median",
    "censo_prop_mat_2ano_biblioteca_sala_leitura_median",
    "log1p_fundeb_receita_contribuicao_median",
    "log1p_fundeb_complementacao_uniao_median",
    "log1p_fundeb_receita_total_median"
]

features_cluster_ausentes = [
    coluna
    for coluna in features_cluster
    if coluna not in base_clusters.columns
]

if features_cluster_ausentes:
    raise ValueError(
        "Features de cluster ausentes: "
        f"{features_cluster_ausentes}"
    )

X_clusters = base_clusters[features_cluster].copy()

display(pd.DataFrame([{
    "municipios_cluster": int(len(X_clusters)),
    "features_cluster": int(X_clusters.shape[1]),
    "valores_ausentes": int(
        X_clusters.isna().sum().sum()
    ),
    "target_real_utilizado": False
}]))


### 6.1 Escolha do número de clusters

Para cada valor de `k`, a imputação por mediana e a padronização são ajustadas sobre o painel municipal. O maior silhouette score indica a separação relativa mais consistente entre os perfis avaliados.


In [0]:
# Objetivo:
#
# Selecionar o número de clusters por silhouette.
#
# Justificativa:
#
# O valor de k não deve ser escolhido apenas por
# conveniência visual.
#
# Ação:
#
# Testa k de 2 a 8 com semente fixa e registra
# inércia e silhouette score.

preparacao_clusters = Pipeline(
    steps=[
        (
            "imputacao",
            SimpleImputer(strategy="median")
        ),
        (
            "padronizacao",
            StandardScaler()
        )
    ]
)

X_clusters_padronizado = (
    preparacao_clusters.fit_transform(X_clusters)
)

resultados_k = []
modelos_k = {}

for k in range(2, 9):
    modelo_k = KMeans(
        n_clusters=k,
        n_init=20,
        random_state=RANDOM_STATE
    )

    rotulos_k = modelo_k.fit_predict(
        X_clusters_padronizado
    )

    resultados_k.append({
        "k": int(k),
        "silhouette": float(
            silhouette_score(
                X_clusters_padronizado,
                rotulos_k
            )
        ),
        "inercia": float(modelo_k.inertia_)
    })

    modelos_k[k] = modelo_k


avaliacao_k_clusters = (
    pd.DataFrame(resultados_k)
    .sort_values("k")
    .reset_index(drop=True)
)

MELHOR_K = int(
    avaliacao_k_clusters.loc[
        avaliacao_k_clusters["silhouette"].idxmax(),
        "k"
    ]
)

display(avaliacao_k_clusters)
print(f"Número de clusters selecionado: {MELHOR_K}")


In [0]:
# Objetivo:
#
# Visualizar o critério de escolha do k.
#
# Justificativa:
#
# O gráfico torna a decisão auditável e comunicável.
#
# Ação:
#
# Plota o silhouette score e destaca o melhor k.

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    avaliacao_k_clusters["k"],
    avaliacao_k_clusters["silhouette"],
    marker="o",
    color="#2E75B6"
)

melhor_linha = avaliacao_k_clusters.loc[
    avaliacao_k_clusters["k"].eq(MELHOR_K)
].iloc[0]

ax.scatter(
    [MELHOR_K],
    [melhor_linha["silhouette"]],
    color="#C44E52",
    s=100,
    zorder=3
)

ax.set_title("Seleção do número de clusters")
ax.set_xlabel("Número de clusters (k)")
ax.set_ylabel("Silhouette score")
ax.set_xticks(avaliacao_k_clusters["k"])
ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig(
    f"{IMAGENS_PATH}/selecao_numero_clusters.png",
    dpi=160,
    bbox_inches="tight"
)
plt.show()


### 6.2 Ajuste final e ordenação dos perfis

Os identificadores originais do KMeans são arbitrários. Para facilitar a interpretação, os clusters serão reordenados pela probabilidade média de risco: o Cluster 1 será o grupo de maior risco relativo.


In [0]:
# Objetivo:
#
# Ajustar o KMeans selecionado e criar rótulos
# interpretáveis por nível relativo de risco.
#
# Justificativa:
#
# Os IDs brutos do KMeans não possuem ordem natural.
#
# Ação:
#
# Prediz clusters, calcula o risco médio de cada grupo
# e remapeia os IDs do maior para o menor risco.

modelo_cluster_final = modelos_k[MELHOR_K]
clusters_brutos = modelo_cluster_final.labels_

base_clusters["cluster_bruto"] = clusters_brutos

ordem_risco_clusters = (
    base_clusters
    .groupby("cluster_bruto")[
        "probabilidade_risco_media"
    ]
    .mean()
    .sort_values(ascending=False)
    .index
    .tolist()
)

mapa_clusters = {
    cluster_bruto: posicao + 1
    for posicao, cluster_bruto in enumerate(
        ordem_risco_clusters
    )
}

base_clusters["cluster"] = (
    base_clusters["cluster_bruto"]
    .map(mapa_clusters)
    .astype(int)
)

base_clusters["perfil_cluster"] = (
    "Cluster "
    + base_clusters["cluster"].astype(str)
    + " — "
    + base_clusters["cluster"].map({
        1: "maior risco relativo",
        MELHOR_K: "menor risco relativo"
    }).fillna("risco intermediário")
)

municipios_clusters = base_clusters.copy()

display(
    municipios_clusters[[
        "co_municipio",
        "no_municipio",
        "sg_uf",
        "quantidade_alunos",
        "probabilidade_risco_media",
        "gap_previsto_meta_pp",
        "cluster",
        "perfil_cluster"
    ]]
    .sort_values([
        "cluster",
        "probabilidade_risco_media"
    ], ascending=[True, False])
    .head(30)
)


### 6.3 Perfil dos clusters

O perfil resume risco, gap, alfabetização, desenvolvimento, vulnerabilidade, infraestrutura e financiamento. As médias descritivas ajudam a nomear os grupos, mas não demonstram causalidade.


In [0]:
# Objetivo:
#
# Descrever as características de cada cluster.
#
# Justificativa:
#
# A clusterização só gera valor quando os grupos podem
# ser interpretados em termos educacionais e sociais.
#
# Ação:
#
# Calcula volumes e médias das principais dimensões.

colunas_perfil = [
    "probabilidade_risco_media",
    "gap_previsto_meta_pp",
    "taxa_alfabetizacao_predita_prob_pct",
    "taxa_alfabetizacao_real_pct",
    "meta_2025_pct",
    "alfabetizacao_2024_pct",
    "atlas_idhm_median",
    "atlas_renda_pc_median",
    "atlas_indice_gini_median",
    "atlas_prop_pobreza_criancas_median",
    "censo_prop_mat_2ano_internet_aprendizagem_median",
    "censo_prop_mat_2ano_alimentacao_median",
    "censo_prop_mat_2ano_biblioteca_sala_leitura_median",
    "fundeb_receita_total_median"
]

perfil_clusters_medias = (
    municipios_clusters
    .groupby(["cluster", "perfil_cluster"])[
        colunas_perfil
    ]
    .mean()
    .reset_index()
)

volumes_clusters = (
    municipios_clusters
    .groupby(["cluster", "perfil_cluster"])
    .agg(
        municipios=("co_municipio", "size"),
        alunos=("quantidade_alunos", "sum")
    )
    .reset_index()
)

perfil_clusters = volumes_clusters.merge(
    perfil_clusters_medias,
    on=["cluster", "perfil_cluster"],
    how="left",
    validate="one_to_one"
)

display(perfil_clusters.sort_values("cluster"))


In [0]:
# Objetivo:
#
# Visualizar os clusters em duas dimensões.
#
# Justificativa:
#
# A projeção PCA permite observar separação e
# sobreposição entre os perfis municipais.
#
# Ação:
#
# Reduz as features padronizadas para dois componentes
# e salva o gráfico de dispersão.

pca = PCA(n_components=2)
coordenadas_pca = pca.fit_transform(
    X_clusters_padronizado
)

visualizacao_pca = pd.DataFrame({
    "componente_1": coordenadas_pca[:, 0],
    "componente_2": coordenadas_pca[:, 1],
    "cluster": municipios_clusters["cluster"].to_numpy(),
    "probabilidade_risco_media": municipios_clusters[
        "probabilidade_risco_media"
    ].to_numpy()
})

fig, ax = plt.subplots(figsize=(10, 7))

sns.scatterplot(
    data=visualizacao_pca,
    x="componente_1",
    y="componente_2",
    hue="cluster",
    palette="tab10",
    size="probabilidade_risco_media",
    sizes=(25, 130),
    alpha=0.75,
    ax=ax
)

ax.set_title(
    "Municípios com padrões semelhantes — projeção PCA"
)
ax.set_xlabel(
    f"Componente 1 ({pca.explained_variance_ratio_[0]:.1%})"
)
ax.set_ylabel(
    f"Componente 2 ({pca.explained_variance_ratio_[1]:.1%})"
)

plt.tight_layout()
plt.savefig(
    f"{IMAGENS_PATH}/clusters_municipais_pca.png",
    dpi=160,
    bbox_inches="tight"
)
plt.show()


### 6.4 Distribuição dos clusters por região

A distribuição regional mostra onde os perfis aparecem com maior frequência. Ela não significa que todos os municípios de uma região sejam iguais.


In [0]:
# Objetivo:
#
# Relacionar os clusters às regiões brasileiras.
#
# Justificativa:
#
# O enunciado solicita identificar regiões com padrões
# semelhantes sem reduzir a análise à divisão geográfica.
#
# Ação:
#
# Mapeia a UF para região e calcula a composição dos
# clusters em cada recorte regional.

mapa_regiao = {
    "11": "Norte", "12": "Norte", "13": "Norte",
    "14": "Norte", "15": "Norte", "16": "Norte",
    "17": "Norte",
    "21": "Nordeste", "22": "Nordeste", "23": "Nordeste",
    "24": "Nordeste", "25": "Nordeste", "26": "Nordeste",
    "27": "Nordeste", "28": "Nordeste", "29": "Nordeste",
    "31": "Sudeste", "32": "Sudeste", "33": "Sudeste",
    "35": "Sudeste",
    "41": "Sul", "42": "Sul", "43": "Sul",
    "50": "Centro-Oeste", "51": "Centro-Oeste",
    "52": "Centro-Oeste", "53": "Centro-Oeste"
}

municipios_clusters["regiao"] = (
    municipios_clusters["co_uf"]
    .astype(str)
    .map(mapa_regiao)
    .fillna("Não identificada")
)

distribuicao_clusters_regiao = (
    municipios_clusters
    .groupby(["regiao", "cluster"])
    .size()
    .rename("municipios")
    .reset_index()
)

distribuicao_clusters_regiao[
    "percentual_na_regiao"
] = (
    distribuicao_clusters_regiao["municipios"]
    / distribuicao_clusters_regiao
    .groupby("regiao")["municipios"]
    .transform("sum")
)

display(
    distribuicao_clusters_regiao
    .sort_values(["regiao", "cluster"])
)

tabela_plot = (
    distribuicao_clusters_regiao
    .pivot(
        index="regiao",
        columns="cluster",
        values="percentual_na_regiao"
    )
    .fillna(0)
    .sort_index()
)

ax = tabela_plot.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 6),
    colormap="tab10"
)

ax.set_title("Composição dos clusters por região")
ax.set_xlabel("")
ax.set_ylabel("Percentual dos municípios da região")
ax.legend(title="Cluster", bbox_to_anchor=(1.02, 1))
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.savefig(
    f"{IMAGENS_PATH}/clusters_por_regiao.png",
    dpi=160,
    bbox_inches="tight"
)
plt.show()


## 7. Síntese estratégica

A síntese consolida os resultados que deverão alimentar o README e o vídeo executivo: cobertura da análise, municípios de maior risco, situação das metas e perfil de cluster mais vulnerável.


In [0]:
# Objetivo:
#
# Produzir uma síntese executiva dinâmica.
#
# Justificativa:
#
# Os achados quantitativos precisam ser apresentados
# sem interpretação causal ou generalização nacional.
#
# Ação:
#
# Resume cobertura, risco, metas e clusters.

municipio_maior_risco = ranking_municipal_risco.iloc[0]
municipio_maior_deficit = municipios_risco_meta.iloc[0]
cluster_maior_risco = perfil_clusters.sort_values(
    "probabilidade_risco_media",
    ascending=False
).iloc[0]

resumo_executivo = pd.DataFrame([{
    "escopo": "municípios do conjunto de teste",
    "municipios_teste": int(len(painel_municipal)),
    "municipios_elegiveis": int(
        len(painel_municipal_elegivel)
    ),
    "municipio_maior_risco": str(
        municipio_maior_risco["rotulo_municipio"]
    ),
    "probabilidade_maior_risco": float(
        municipio_maior_risco[
            "probabilidade_risco_media"
        ]
    ),
    "municipio_maior_deficit_meta": str(
        municipio_maior_deficit["rotulo_municipio"]
    ),
    "maior_deficit_meta_pp": float(
        municipio_maior_deficit[
            "gap_previsto_meta_pp"
        ]
    ),
    "municipios_abaixo_meta": int(
        municipios_risco_meta[
            "gap_previsto_meta_pp"
        ].lt(0).sum()
    ),
    "quantidade_clusters": int(MELHOR_K),
    "cluster_maior_risco": str(
        cluster_maior_risco["perfil_cluster"]
    ),
    "modelo_retreinado": False,
    "limiar_alterado": False
}])

display(resumo_executivo)

print("CONCLUSÃO EXECUTIVA")
print("=" * 80)
print(
    f"A análise cobre {len(painel_municipal):,} municípios "
    "não utilizados no treinamento."
)
print(
    "Município elegível de maior risco previsto: "
    f"{municipio_maior_risco['rotulo_municipio']} "
    f"({municipio_maior_risco['probabilidade_risco_media']:.2%})."
)
print(
    "Município com maior déficit previsto para a meta: "
    f"{municipio_maior_deficit['rotulo_municipio']} "
    f"({municipio_maior_deficit['gap_previsto_meta_pp']:.2f} p.p.)."
)
print(
    f"O KMeans identificou {MELHOR_K} perfis municipais."
)
print(
    "Os resultados apoiam priorização e investigação, "
    "mas não representam causalidade nem um ranking "
    "nacional completo."
)


## 8. Persistência dos artefatos

Serão salvas apenas tabelas agregadas por município ou cluster. Nenhum identificador individual de aluno será incluído nos produtos deste notebook.


In [0]:
# Objetivo:
#
# Persistir os resultados estratégicos na Fase 3.
#
# Justificativa:
#
# Artefatos agregados permitem auditoria e comunicação
# sem expor identificadores individuais.
#
# Ação:
#
# Salva rankings, gaps, clusters e síntese executiva.

artefatos_aplicacao = {
    "painel_municipal_teste.csv": painel_municipal,
    "ranking_municipal_risco.csv": ranking_municipal_risco,
    "municipios_risco_meta.csv": municipios_risco_meta,
    "avaliacao_k_clusters.csv": avaliacao_k_clusters,
    "perfil_clusters.csv": perfil_clusters,
    "municipios_clusters.csv": municipios_clusters,
    "distribuicao_clusters_regiao.csv": (
        distribuicao_clusters_regiao
    ),
    "resumo_executivo_aplicacao.csv": resumo_executivo
}

arquivos_salvos = []

for nome_arquivo, dados in artefatos_aplicacao.items():
    caminho = f"{APLICACAO_PATH}/{nome_arquivo}"
    dados.to_csv(
        caminho,
        sep=";",
        encoding="utf-8-sig",
        index=False
    )
    arquivos_salvos.append({
        "artefato": nome_arquivo,
        "caminho": caminho,
        "registros": int(len(dados))
    })

manifesto_artefatos = pd.DataFrame(arquivos_salvos)

display(manifesto_artefatos)


## 9. Validação final e critérios de encerramento

O notebook será considerado concluído quando:

1. os 834 municípios do teste estiverem representados no painel;
2. nenhum aluno individual aparecer nos artefatos agregados;
3. o ranking utilizar um mínimo explícito de alunos;
4. a escala da meta tiver sido validada;
5. o número de clusters tiver sido escolhido pelo silhouette score;
6. o target real não tiver sido usado como feature da clusterização;
7. todas as saídas estiverem na Fase 3;
8. modelo e limiar permanecerem inalterados.


In [0]:
# Objetivo:
#
# Validar o encerramento da aplicação estratégica.
#
# Justificativa:
#
# A conclusão depende de contratos verificáveis e não
# apenas da exibição de gráficos.
#
# Ação:
#
# Confirma cobertura, privacidade, persistência e
# imutabilidade do modelo.

arquivos_esperados = [
    f"{APLICACAO_PATH}/{nome}"
    for nome in artefatos_aplicacao
]

validacao_final = pd.DataFrame([{
    "municipios_painel": int(len(painel_municipal)),
    "municipios_esperados": int(
        previsoes["co_municipio"].nunique()
    ),
    "cobertura_municipal_ok": bool(
        len(painel_municipal)
        == previsoes["co_municipio"].nunique()
    ),
    "ids_aluno_em_artefatos_agregados": False,
    "target_real_em_features_cluster": bool(
        "taxa_alfabetizacao_real_pct"
        in features_cluster
    ),
    "modelo_original": str(
        metadados_modelo["modelo"]
    ),
    "limiar_original": float(
        metadados_modelo["limiar"]
    ),
    "modelo_retreinado": False,
    "limiar_alterado": False,
    "arquivos_esperados": int(len(arquivos_esperados)),
    "arquivos_existentes": int(sum(
        Path(caminho).exists()
        for caminho in arquivos_esperados
    ))
}])

if not validacao_final.loc[0, "cobertura_municipal_ok"]:
    raise RuntimeError(
        "O painel não representa todos os municípios do teste."
    )

if validacao_final.loc[0, "target_real_em_features_cluster"]:
    raise RuntimeError(
        "O target real foi incluído indevidamente nos clusters."
    )

if (
    validacao_final.loc[0, "arquivos_existentes"]
    != validacao_final.loc[0, "arquivos_esperados"]
):
    raise RuntimeError(
        "Nem todos os artefatos estratégicos foram salvos."
    )

display(validacao_final)

print(
    "Aplicação estratégica concluída com ranking "
    "municipal, análise de metas e clusterização."
)
